# G2L-Theorie mit NN-Emulator

Pipeline: NN (Parameter -> $P_{gm}(k, z=0)$) -> Wachstumsfaktor-Skalierung -> Limber $C_{\kappa g}(\ell)$ -> Hankel $\gamma_t(\theta)$.

Rechnung in `g2l_nn.py` (Emulator: `gm_emulator.npz`, erzeugt mit `export_gm_emulator.py`).
**Vorlaeufig (Variante A):** das NN kennt nur $z=0$, die z-Abhaengigkeit ist $D^2(z)$. Das ist fuer die Lens-Rotverschiebungen nur grob (10-100 % Fehler in $\gamma_t$). Austausch nur in `G2LTheory.pk_matrix`.

In [ ]:
import sys
sys.path.insert(0, '/home/rhoffmann/MA/Modelling')
import numpy as np
import matplotlib.pyplot as plt
import g2l_nn as g

data, theory, space, post = g.build()
zs, n_lenses, n_sources = theory.zs, theory.n_lenses, theory.n_sources
print(len(data.combos), 'Kombinationen,', data.n, 'Datenpunkte,', space.ndim, 'Parameter')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, n in n_lenses.items():
    axes[0].plot(zs, n, label=name)
for k, n in n_sources.items():
    axes[1].plot(zs, n, label=f'S{k}')
axes[0].set_title('Lens n(z)'); axes[1].set_title('Source n(z)')
for ax in axes:
    ax.set_xlabel('z'); ax.legend(fontsize=7)
plt.show()

In [ ]:
# Theorie am Startpunkt (alte Einzelkonfiguration); nach dem MCMC siehe CompareG2L.ipynb
theta0 = space.fiducial()
model0 = theory.model_vector(theta0, space)
print('chi2 / N =', round(data.chi2(model0), 1), '/', data.n)
print('Parameter:'); [print(f'  {n:28s} {v:8.3f}  [{lo:g}, {hi:g}]') for n, v, lo, hi in zip(space.names[:10], theta0, space.lo, space.hi)]

In [ ]:
for lens in space.lens_names:
    items = [(i, c) for i, c in enumerate(data.combos) if c[0] == lens]
    fig, axes = plt.subplots(1, len(items), figsize=(3.6*len(items), 3.4), sharey=True, squeeze=False)
    for ax, (i, (ln, sb)) in zip(axes[0], items):
        sl = data.slices[i]
        th = data.theta[i]
        ax.errorbar(th, th*data.gt[i], yerr=th*data.sig_diag[i], fmt='o', ms=3, capsize=2, color='k')
        ax.plot(th, th*model0[sl], color='tab:red')
        ax.set_xscale('log'); ax.set_title(f'{ln} x S{sb}', fontsize=9); ax.set_xlabel(r'$\theta$ [arcmin]')
    axes[0, 0].set_ylabel(r'$\theta\,\gamma_t$ [arcmin]')
    plt.tight_layout(); plt.show()